# **Getting Started with kmeans_tpu**

`kmeans_tpu` is a JAX-based Python package implementing K-Means and K-Means++
clustering optimised for Google Tensor Processing Units (TPUs).

This notebook walks you through the basic usage of the package, from
installation to your first clustering result.

> **Note:** This notebook requires a TPU runtime.
> In Colab: Runtime → Change runtime type → TPU v5e-1 TPU

> **Compatibility:** This notebook has been tested with JAX 0.7.2 and
> Python 3.13 on Google Colab.

## 1. Installation

First, install the package directly from GitHub:

In [ ]:
!pip install git+https://github.com/vbrachetta/kmeans_tpu.git

## 2. Verify the TPU is available

Before running any code, verify that JAX can see the TPU:

In [ ]:
import jax
print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")

## 3. Generate some data

We generate a simple synthetic dataset using NumPy — 1,000 samples with
4 features. In practice, you would replace this with your own data.

> **Important:** Data must be float32 or bfloat16 for TPU compatibility.



In [ ]:
import numpy as np
import jax.numpy as jnp

np.random.seed(42)
data = jnp.array(np.random.randn(1_000, 4).astype("float32"))

print(f"Data shape : {data.shape}")
print(f"Data dtype : {data.dtype}")
print(f"Data device: {data.devices()}")

## 4. Run K-Means clustering

We cluster the data into 3 groups using the default K-Means++ initialisation.
The `verbose=True` option prints the convergence progress every 10 iterations.

In [ ]:
from kmeans_tpu import kmeans

centroids, assignments, n_iter = kmeans(data, k=3, verbose=True)

print(f"\nConverged in {n_iter} iterations")
print(f"Centroids shape   : {centroids.shape}")
print(f"Assignments shape : {assignments.shape}")

## 5. Inspect the results

Let us look at the cluster assignments and how many points belong to each cluster.

In [ ]:
import numpy as np

counts = np.bincount(np.array(assignments), minlength=3)
for i, count in enumerate(counts):
    print(f"Cluster {i}: {count} points")

## 6. Evaluate clustering quality

The inertia (within-cluster sum of squares) measures how tight the clusters are.
Lower values indicate better clustering.

In [ ]:
from kmeans_tpu import inertia

score = inertia(data, centroids, assignments)
print(f"Inertia: {score:.2f}")

## 7. Try random initialisation

By default, `kmeans_tpu` uses K-Means++ initialisation, which generally produces
better results than random initialisation. Let us compare the two:

In [ ]:
centroids_rnd, assignments_rnd, n_iter_rnd = kmeans(data, k=3, init="random", verbose=True)
score_rnd = inertia(data, centroids_rnd, assignments_rnd)

centroids_pp, assignments_pp, n_iter_pp = kmeans(data, k=3, init="plusplus", verbose=True)
score_pp = inertia(data, centroids_pp, assignments_pp)

print(f"\nRandom    — Inertia: {score_rnd:.2f}, Iterations: {n_iter_rnd}")
print(f"K-Means++ — Inertia: {score_pp:.2f}, Iterations: {n_iter_pp}")

## 8. Using your own data

To use your own data, simply convert it to a JAX array with float32 dtype:
```python
import jax.numpy as jnp
import numpy as np
from kmeans_tpu import kmeans

data = jnp.array(np.load("your_data.npy").astype("float32"))

centroids, assignments, n_iter = kmeans(
    data,
    k=8,
    init="plusplus",
    n_iter=100,
    tol=1e-4,
    seed=42,
    verbose=True,
)
print(f"Converged in {n_iter} iterations")
```

## Next steps

- See `02_kmeans_vs_kmeanspp.ipynb` for a visual comparison of both initialisation strategies
- See `03_bfloat16.ipynb` for a demonstration of bfloat16 support